# 07 - Final Regression (Day 5)

Compares the current Bronze, Silver and Gold outputs with the **UAT baseline**. The baseline was frozen from job run 42362058080150 (2026-11-04), where all six tasks succeeded and 117/117 validation checks passed.

It checks:
1. Row counts of every Bronze, Silver, DQ and Gold table
2. DQ outcome: rejected records by reason, warning records, rules
3. Business KPI totals (latency, availability, packet loss, throughput, outage minutes, outage impact, customers)
4. Latest pipeline validation summaries: all three layers present, 0 FAIL

It is read-only: it writes nothing. If any check fails, the last cell raises an error.

The baseline matches the current landing data. When new source files arrive, the counts are expected to change. Re-baseline then by updating `BASELINE` after the new run passes `06_Pipeline_Validation` and UAT.

No `dbutils` is used.

In [0]:
# 1. UAT baseline (frozen from job run 42362058080150, 2026-11-04)
from pyspark.sql import functions as F

catalog = "capstone6_telecom"
B, S, G = f"{catalog}.bronze", f"{catalog}.silver", f"{catalog}.gold"
TOLERANCE = 1e-3   # absolute tolerance for KPI averages / totals

BASELINE_COUNTS = {
    # Bronze (one loaded version per master key, ADLS feeds = landing files)
    f"{B}.bronze_tower_master": 161, f"{B}.bronze_network_topology_master": 170, f"{B}.bronze_region_lookup": 179,
    f"{B}.bronze_network_kpi_measurements": 11894, f"{B}.bronze_network_outage_events": 696,
    f"{B}.bronze_customer_complaints": 354,
    # Silver
    f"{S}.silver_tower_master": 161, f"{S}.silver_network_topology_master": 170, f"{S}.silver_region_lookup": 179,
    f"{S}.silver_network_kpi_measurements": 11829, f"{S}.silver_network_outage_events": 696,
    f"{S}.silver_customer_complaints": 354,
    f"{S}.silver_rejected_records": 65, f"{S}.silver_warning_records": 36,
    f"{S}.silver_dq_rule_catalogue": 129, f"{S}.silver_dq_rule_results": 129,
    # Gold
    f"{G}.dim_tower": 161, f"{G}.dim_region": 179, f"{G}.dim_cell": 170,
    f"{G}.fact_network_kpi": 11829, f"{G}.fact_network_outage": 696, f"{G}.fact_customer_complaint": 354,
    f"{G}.fact_outage_impact": 696, f"{G}.gold_network_quality_daily": 3998, f"{G}.gold_region_sla_summary": 179,
}

# Rejected records by reason, warning records by reason
BASELINE_REJECTS = {"EXACT_DUPLICATE_ROW": 43, "NEGATIVE_VALUE": 22}
BASELINE_WARNINGS = {"IQR_OUTLIER": 26, "MISSING_OPTIONAL_FIELD": 10}

# Business KPI totals on Gold (SQL expression -> baseline value)
BASELINE_KPIS = {
    "Avg latency ms (outliers excluded)": (f"SELECT AVG(latency_ms) FROM {G}.fact_network_kpi WHERE NOT is_iqr_outlier", 84.5821),
    "Avg availability pct": (f"SELECT AVG(availability_pct) FROM {G}.fact_network_kpi", 97.5042),
    "Avg packet loss pct": (f"SELECT AVG(packet_loss_pct) FROM {G}.fact_network_kpi", 2.4822),
    "Total throughput mbps": (f"SELECT SUM(throughput_mbps) FROM {G}.fact_network_kpi", 5334976.97),
    "Total outage minutes": (f"SELECT SUM(outage_duration_minutes) FROM {G}.fact_network_outage", 3480.0),
    "Outages with telemetry evidence": (f"SELECT COUNT_IF(has_telemetry_evidence) FROM {G}.fact_outage_impact", 145),
    "Outages with customer impact": (f"SELECT COUNT_IF(has_customer_impact) FROM {G}.fact_outage_impact", 51),
    "Complaining customers": (f"SELECT COUNT(DISTINCT customer_id) FROM {G}.fact_customer_complaint", 354),
}

results = []   # (area, check, baseline, actual, status)

def record(area, name, baseline, actual, tol=0.0):
    ok = actual is not None and abs(float(actual) - float(baseline)) <= tol
    results.append((area, name, str(baseline), str(actual), "PASS" if ok else "FAIL"))

In [0]:
# 2. Row counts and DQ outcome
for table, expected in BASELINE_COUNTS.items():
    record("ROW_COUNT", table.split(".", 1)[1], expected, spark.table(table).count())

rej = {r.reject_reason: r.n for r in spark.table(f"{S}.silver_rejected_records")
       .groupBy("reject_reason").agg(F.count("*").alias("n")).collect()}
for reason in sorted(set(BASELINE_REJECTS) | set(rej)):
    record("DQ_OUTCOME", f"Rejected: {reason}", BASELINE_REJECTS.get(reason, 0), rej.get(reason, 0))

warn = {r.warning_reason: r.n for r in spark.table(f"{S}.silver_warning_records")
        .groupBy("warning_reason").agg(F.count("*").alias("n")).collect()}
for reason in sorted(set(BASELINE_WARNINGS) | set(warn)):
    record("DQ_OUTCOME", f"Warning: {reason}", BASELINE_WARNINGS.get(reason, 0), warn.get(reason, 0))

In [0]:
# 3. Business KPI totals (rounded to the baseline precision)
for name, (sql, expected) in BASELINE_KPIS.items():
    actual = spark.sql(sql).first()[0]
    decimals = len(str(expected).split(".")[1]) if isinstance(expected, float) else 0
    record("BUSINESS_KPI", name, expected, round(float(actual), decimals) if actual is not None else None, TOLERANCE)

In [0]:
# 4. Latest pipeline validation summaries (written by tests/06_Pipeline_Validation)
summary = spark.sql(f"""
    SELECT * FROM {B}.bronze_validation_summary UNION ALL
    SELECT * FROM {S}.silver_validation_summary UNION ALL
    SELECT * FROM {G}.gold_validation_summary""")
by_layer = {r.layer: r for r in summary.groupBy("layer").agg(
    F.count("*").alias("checks"), F.count_if(F.col("status") == "FAIL").alias("fails"),
    F.max("validated_at").alias("validated_at")).collect()}
for layer in ["BRONZE", "SILVER", "GOLD"]:
    r = by_layer.get(layer)
    record("VALIDATION", f"{layer} validation summary present (checks > 0)", 1, 1 if r and r.checks > 0 else 0)
    record("VALIDATION", f"{layer} blocking checks failed", 0, r.fails if r else None)
print({k: (v.checks, str(v.validated_at)) for k, v in by_layer.items()})

In [0]:
# 5. Regression report
report = spark.createDataFrame(results, "area string, check_name string, baseline string, actual string, status string")
display(report.groupBy("area").pivot("status", ["PASS", "FAIL"]).count().fillna(0).orderBy("area"))
display(report.orderBy(F.expr("CASE status WHEN 'FAIL' THEN 0 ELSE 1 END"), "area", "check_name"))

failed = [r for r in results if r[-1] == "FAIL"]
print(f"Regression checks: {len(results)} | PASS: {len(results) - len(failed)} | FAIL: {len(failed)}")
assert not failed, "Regression failed: " + "; ".join(f"{r[1]} (baseline {r[2]}, actual {r[3]})" for r in failed[:10])